# 🧹 Fase 3: Limpieza, Transformación e Ingeniería de Características
## Proyecto de Ciencia de Datos - Inside Airbnb NYC

---
### 🎯 Objetivos de esta fase:
1. Implementar el pipeline de limpieza y parseo para precios, baños y textos anidados.
2. Aplicar imputaciones semánticas justificadas sobre variables críticas (`bedrooms`, `beds`, `accommodates`).
3. Construir variables derivadas con alto valor predictivo: distancias geodésicas (Haversine) a puntos turísticos, ratios de capacidad, indicadores de comodidades (`amenities`).
4. Generar el dataset listo para modelado en `data/selected/airbnb/processed/nyc_features.parquet`.

In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_listings
from src.cleaning import clean_airbnb_dataset
from src.features import engineer_features

print("✅ Módulos importados correctamente.")

✅ Módulos importados correctamente.


---
## 1. Carga y Ejecución del Pipeline de Limpieza

In [13]:
# 1. Carga cruda
df_raw = load_raw_listings(city="nyc")
print(f"Dataset crudo: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.")

# 2. Pipeline de Limpieza
df_clean = clean_airbnb_dataset(df_raw, min_price=10.0, max_price=2000.0)
print(f"Dataset tras limpieza y filtrado de outliers: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas.")

📖 Cargando /home/cathe/Documents/projects/ciencia_datos_10Semestre/project_Inside_Airbnb/data-science/data/selected/airbnb/raw/nyc/listings.csv.gz...
📊 Dataset cargado: 30,259 filas x 90 columnas.
Dataset crudo: 30,259 filas x 90 columnas.
Dataset tras limpieza y filtrado de outliers: 21,347 filas x 122 columnas.


---
## 2. Ingeniería de Características (Feature Engineering)
Agregamos distancias geodésicas a Times Square, Central Park, Financial District, ratios de infraestructura y métricas de reseñas.

In [14]:
# 3. Ingeniería de Características
df_features = engineer_features(df_clean)
print(f"Dataset final con características: {df_features.shape[0]:,} filas x {df_features.shape[1]} columnas.")

# Muestra de nuevas columnas generadas
feat_cols = [c for c in df_features.columns if 'dist_' in c or 'ratio' in c or 'has_' in c]
print(f"Total de características generadas: {len(feat_cols)}")
df_features[['price_usd', 'log_price', 'dist_times_square_km', 'accommodates_per_bedroom', 'total_amenities_count']].head(5)

Dataset final con características: 21,347 filas x 136 columnas.
Total de características generadas: 36


,price_usd,log_price,dist_times_square_km,accommodates_per_bedroom,total_amenities_count
0,113.97,4.744671,12.511430,2.0,49
1,117.27,4.772970,6.047341,1.5,26
2,80.06,4.395190,6.001197,1.0,18
3,77.17,4.358886,4.599598,1.0,22
4,202.47,5.315519,7.410405,2.0,50


---
## 3. Exportación del Dataset Procesado a Formato Parquet

In [15]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "selected" / "airbnb" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_parquet = PROCESSED_DIR / "nyc_listings_features.parquet"

df_features.to_parquet(output_parquet, index=False)
print(f"💾 Dataset guardado exitosamente en: {output_parquet}")

💾 Dataset guardado exitosamente en: /home/cathe/Documents/projects/ciencia_datos_10Semestre/project_Inside_Airbnb/data-science/data/selected/airbnb/processed/nyc_listings_features.parquet
